### Import Library 


In [1]:
import cv2, torch
import numpy as np 
import torch.nn as nn
import torchvision
# import torchvision.models as tv_models
from torchvision.models.resnet import ResNet, BasicBlock
from sklearn.cluster import KMeans

In [2]:
class Anchors:
    def __init__(self, boxes: np.array, n_clusters: int = 5):
        self.boxes = boxes
        self.n_clusters = n_clusters
        self.cluster_model = KMeans(
            n_clusters=self.n_clusters, n_init=30, max_iter=1000)

    def get(self):
        wh = self.get_boxes_wh()
        self.cluster_model.fit(wh)
        return self.cluster_model.cluster_centers_

    def get_boxes_wh(self):
        box_wh = self.boxes[:, 2:]
        return box_wh

In [3]:
class Upsample(nn.Module):
    def __init__(self, scale_factor, mode='nearest'):
        super(Upsample, self).__init__()

        self.interp = nn.functional.interpolate
        self.scale_factor = scale_factor
        self.mode = mode

    def forward(self, x, target_size=None):
        # If target_size provided, resize to that HxW to avoid off-by-one
        # mismatches from repeated down/up sampling. Otherwise use scale_factor.
        if target_size is not None:
            x = self.interp(x, size=target_size, mode=self.mode)
        else:
            x = self.interp(x, scale_factor=self.scale_factor, mode=self.mode)
        return x

In [4]:
class FeatureMapper(ResNet):
    def __init__(self, input_channels, n_anchors_per_scale, n_classes, pretrained=True):
        super(FeatureMapper, self).__init__(BasicBlock, [2, 2, 2, 2])

        self.input_channels = input_channels
        self.n_anchors_per_scale = n_anchors_per_scale
        self.n_classes = n_classes
        self.output_channels = self.n_anchors_per_scale*(5 + self.n_classes)

        # init resnet18 weights
        if pretrained == True and self.input_channels == 3:
            self.load_state_dict(torchvision.models.resnet18(
                pretrained=True).state_dict())

        self.conv1 = nn.Conv2d(
            self.input_channels, 64,
            kernel_size=(7, 7),
            stride=(2, 2),
            padding=(3, 3),
            bias=False)

        self.conv2 = nn.Conv2d(
            64, 64,
            kernel_size=(7, 7),
            stride=(2, 2),
            padding=(3, 3),
            bias=False)

        sm_c, md_c, lg_c = 512, 256, 128

        self.lg_fmapper = nn.Conv2d(
            sm_c, self.output_channels,
            kernel_size=(1, 1),
            stride=(1, 1),
            bias=False)

        self.md_fmapper = nn.Conv2d(
            sm_c+md_c, self.output_channels,
            kernel_size=(1, 1),
            stride=(1, 1),
            bias=False)

        self.sm_fmapper = nn.Conv2d(
            sm_c+md_c+lg_c, self.output_channels,
            kernel_size=(1, 1),
            stride=(1, 1),
            bias=False)

        self.upsampler = Upsample(2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)

        addon_1 = self.layer2(x)
        addon_2 = self.layer3(addon_1)
        

        fmap = self.layer4(addon_2)
        lg_scale_pred = self.lg_fmapper(fmap)

        fmap = self.upsampler(fmap, target_size=addon_2.shape[2:])
        fmap = torch.cat((fmap, addon_2), dim=1)
        md_scale_pred = self.md_fmapper(fmap)

        fmap = self.upsampler(fmap, target_size=addon_1.shape[2:])
        fmap = torch.cat((fmap, addon_1), dim=1)
        sm_scale_pred = self.sm_fmapper(fmap)

        return lg_scale_pred, md_scale_pred, sm_scale_pred

In [5]:
input_channels = 3
n_anchors = 3
n_classes = 4

feature_mapper = FeatureMapper(input_channels, n_anchors, n_classes)

random_input = torch.randn(5, 3, 416, 416)

lg_scale_pred, md_scale_pred, sm_scale_pred = feature_mapper(random_input)

print(
    f' large grid scale feature map size: {lg_scale_pred.shape}\n\n',
    f'medium grid scale feature map size: {md_scale_pred.shape}\n\n',
    f'small grid scale feature map size: {sm_scale_pred.shape}')

/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


 large grid scale feature map size: torch.Size([5, 27, 13, 13])

 medium grid scale feature map size: torch.Size([5, 27, 26, 26])

 small grid scale feature map size: torch.Size([5, 27, 52, 52])


In [6]:
class BoundingBoxPredictor(nn.Module):
    def __init__(self, anchors):
        super(BoundingBoxPredictor, self).__init__()
        self.anchors = anchors
        self.n_anchors_per_scale = len(self.anchors)

    def forward(self, feature_map):
        feature_map = torch.permute(feature_map, (0, 2, 3, 1))
        r"""
        #feature_map size: (N, c, h, w) --> (N, h, w, c)
        #1. feature_map size: (N, h, w, c) --> (N, h, w, n_anchor, bb_coord+cofidence+class_scores)
        #2. obj_score size: (N, h, w, n_anchor, cofidence)
        #3. pred_coordinates size: (N, h, w, n_anchor, bb_coord)
        #4. class_scores size: (N, h, w, n_anchor, class_scores)
        """

        fmap_shape = feature_map.shape
        feature_map = feature_map.reshape(
            *fmap_shape[0:3], self.n_anchors_per_scale, -1)
        obj_scores = feature_map[..., 0].unsqueeze(dim=-1)
        boxlocs = feature_map[..., 1:5]
        class_scores = feature_map[..., 5:]

        obj_scores = torch.sigmoid(obj_scores)

        idx = self.getOffset(fmap_shape[1:3])
        boxXY = torch.sigmoid(boxlocs[..., :2]) + idx
        boxWH = torch.exp(boxlocs[..., 2:4]) * self.anchors

        class_scores = torch.sigmoid(class_scores)

        bboxes = torch.cat((boxXY, boxWH), dim=-1)
        predictions = torch.cat((obj_scores, bboxes, class_scores), dim=-1)
        predictions = predictions.reshape(
            predictions.shape[0], -1, predictions.shape[-1])
        return predictions

    def getOffset(self, shape):
        h, w = shape
        hindex = torch.Tensor([i for i in range(0, h)]).unsqueeze(dim=-1)
        windex = torch.Tensor([i for i in range(0, w)]).unsqueeze(dim=0)
        hindex = hindex.tile(1, w)
        windex = windex.tile(h, 1)
        idx = torch.stack((windex, hindex), dim=-1).reshape(1, h, w, 1, 2)
        return idx

In [7]:
rand_anchors = torch.randn(3, 2)
bbox_predictor = BoundingBoxPredictor(rand_anchors)

bboxes = bbox_predictor(lg_scale_pred)

print(f'bounding box prediction shape: {bboxes.shape}')

bounding box prediction shape: torch.Size([5, 507, 9])


In [8]:
class YOLOv8(nn.Module):
    def __init__(self, input_channels, anchors, n_classes):
        super(YOLOv8, self).__init__()

        self.input_channels = input_channels
        self.anchors = anchors
        self.n_anchors_per_scale = len(self.anchors)//3
        self.n_classes = n_classes

        self.feature_mapper = FeatureMapper(
            self.input_channels, self.n_anchors_per_scale, self.n_classes)
        self.sm_box_predictor = BoundingBoxPredictor(self.anchors[:3])
        self.md_box_predictor = BoundingBoxPredictor(self.anchors[3:6])
        self.lg_box_predictor = BoundingBoxPredictor(self.anchors[6:9])

    def forward(self, x):
        lg_scale_pred, md_scale_pred, sm_scale_pred = self.feature_mapper(x)
        lg_scale_pred = self.lg_box_predictor(lg_scale_pred)
        md_scale_pred = self.md_box_predictor(md_scale_pred)
        sm_scale_pred = self.sm_box_predictor(sm_scale_pred)
        pred_bbox = torch.cat(
            (lg_scale_pred, md_scale_pred, sm_scale_pred), dim=1)

        return pred_bbox

In [9]:
input_channels = 3
n_anchors = 9
n_classes = 4

bounding_boxes = np.random.randn(1000, 4)
anchor_obj = Anchors(bounding_boxes, n_anchors)
anchor_boxes = torch.from_numpy(anchor_obj.get())

yolo_model = YOLOv8(input_channels, anchor_boxes, n_classes)

print(f'Model Neural Network Architecture: \n\n{yolo_model}')

Model Neural Network Architecture: 

YOLOv8(
  (feature_mapper): FeatureMapper(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(6

/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
random_data = torch.randn(2, 3, 416, 416)

pred_bboxes = yolo_model(random_data)
print(f'bounding box prediction size: {pred_bboxes.shape}')

bounding box prediction size: torch.Size([2, 10647, 9])


In [11]:
class BoundingBoxPostProcessing:
    def __init__(self, bboxes: torch.Tensor):
        self.bboxes = bboxes

    def filter_boxes(self, scoreThresh: float = 0.5):
        N = self.bboxes.shape[0]

        obj_scores = self.bboxes[..., 0]
        boxlocs = self.bboxes[..., 1:5]
        class_scores = self.bboxes[..., 5:]

        # confidence_score: multiply class probability scores with objectness score
        confidence_scores = class_scores * obj_scores.unsqueeze(dim=-1)
        confidence_scores, _ = confidence_scores.max(dim=-1)

        # get index of the class with highest probability
        classes = class_scores.argmax(dim=-1)

        new_confidence_scores = []
        new_boxlocs = []
        new_classes = []

        for batch in range(N):
            sample_confidence_scores = confidence_scores[batch]
            sample_boxlocs = boxlocs[batch]
            sample_classes = classes[batch]

            mask = sample_confidence_scores >= scoreThresh

            new_confidence_scores.append(sample_confidence_scores[mask])
            new_boxlocs.append(sample_boxlocs[mask])
            new_classes.append(sample_classes[mask])

        return tuple(new_confidence_scores), tuple(new_boxlocs), tuple(new_classes)

    def non_max_supression(self, confidence_scores: tuple, boxlocs: tuple,
                           classes: tuple, iou_threshold: float = 0.5):
        N = len(confidence_scores)
        new_confidence_scores = []
        new_boxlocs = []
        new_classes = []

        for batch in range(N):
            sample_confidence_scores = confidence_scores[batch]
            sample_boxlocs = boxlocs[batch]
            sample_classes = classes[batch]

            idx = torchvision.ops.nms(
                sample_boxlocs, sample_confidence_scores, iou_threshold)

            sample_confidence_scores = torch.index_select(
                sample_confidence_scores, 0, idx)
            sample_boxlocs = torch.index_select(sample_boxlocs, 0, idx)
            sample_classes = torch.index_select(sample_classes, 0, idx)

            new_confidence_scores.append(sample_confidence_scores)
            new_boxlocs.append(sample_boxlocs)
            new_classes.append(sample_classes)

        return tuple(new_confidence_scores), tuple(new_boxlocs), tuple(new_classes)

In [12]:
bbox_processor = BoundingBoxPostProcessing(pred_bboxes)
confidence_scores, boxlocs, classes = bbox_processor.filter_boxes()
confidence_scores, boxlocs, classes = bbox_processor.non_max_supression(
    confidence_scores, boxlocs, classes)

print(
    f'bounding box confidence scores: {confidence_scores[0].shape} \n\n',
    f'bounding box coord: {boxlocs[0].shape} \n\n',
    f'bounding box classes: {classes[0].shape}')

bounding box confidence scores: torch.Size([467]) 

 bounding box coord: torch.Size([467, 4]) 

 bounding box classes: torch.Size([467])


In [13]:

import os
import glob
import xml.etree.ElementTree as ET
from PIL import Image


class FruitDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir="images", ann_dir="annotations", classes=None, transforms=None):
        self.images_dir = images_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        # collect annotation files
        xml_paths = sorted(glob.glob(os.path.join(self.ann_dir, "*.xml")))
        self.samples = []  # list of (img_path, [(label_name, [xmin,ymin,xmax,ymax]), ...])

        for xp in xml_paths:
            try:
                tree = ET.parse(xp)
                root = tree.getroot()
            except Exception:
                continue

            # get filename from XML; fall back to xml basename if missing
            filename_tag = root.find('filename')
            if filename_tag is None or filename_tag.text is None:
                img_name = os.path.splitext(os.path.basename(xp))[0] + '.jpg'
            else:
                img_name = filename_tag.text

            img_path = os.path.join(self.images_dir, img_name)
            if not os.path.exists(img_path):
                # try png alternative
                alt = os.path.splitext(img_path)[0] + '.png'
                if os.path.exists(alt):
                    img_path = alt
                else:
                    # image missing, skip
                    continue

            objs = []
            for obj in root.findall('object'):
                name_tag = obj.find('name')
                bnd = obj.find('bndbox')
                if name_tag is None or bnd is None:
                    continue
                try:
                    xmin = int(float(bnd.find('xmin').text))
                    ymin = int(float(bnd.find('ymin').text))
                    xmax = int(float(bnd.find('xmax').text))
                    ymax = int(float(bnd.find('ymax').text))
                except Exception:
                    continue
                objs.append((name_tag.text, [xmin, ymin, xmax, ymax]))

            if len(objs) > 0:
                self.samples.append((img_path, objs))

        # build class -> index mapping if not provided
        if classes is None:
            class_names = sorted({n for _, objs in self.samples for n, _ in objs})
            self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        else:
            self.class_to_idx = {c: i for i, c in enumerate(classes)}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, objs = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        # convert to tensor (C,H,W) float in [0,1]
        img_t = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0

        boxes = []
        labels = []
        for name, bbox in objs:
            boxes.append(bbox)
            labels.append(self.class_to_idx.get(name, -1))

        boxes_t = torch.tensor(boxes, dtype=torch.long) if len(boxes) > 0 else torch.zeros((0, 4), dtype=torch.long)
        labels_t = torch.tensor(labels, dtype=torch.long) if len(labels) > 0 else torch.zeros((0,), dtype=torch.long)

        sample = {
            'image': img_t,
            'boxes': boxes_t,
            'labels': labels_t,
            'path': img_path,
        }

        if self.transforms is not None:
            sample = self.transforms(sample)

        return sample


# --- demo / quick check ---
if __name__ == '__main__':
    ds = FruitDataset(images_dir='images', ann_dir='annotations')
    print(f'Found {len(ds)} samples')
    if len(ds) > 0:
        s = ds[0]
        print('example image tensor shape:', s['image'].shape)
        print('example boxes shape:', s['boxes'].shape)
        print('example labels shape:', s['labels'].shape)

Found 200 samples
example image tensor shape: torch.Size([3, 300, 400])
example boxes shape: torch.Size([3, 4])
example labels shape: torch.Size([3])
